# 05 — KPI Framework & Tableau Data Preparation

## Objective
Define and compute Key Performance Indicators (KPIs), then export Tableau-ready datasets for dashboard creation.

**Deliverables:**
1. KPI definitions and computed values
2. `tableau_main.csv` — Full cleaned dataset
3. `tableau_kpi_summary.csv` — Pre-aggregated KPIs
4. `tableau_segment_churn.csv` — Churn rates by segment
5. `tableau_cluster_profiles.csv` — K-Means cluster profiles

---

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/telco_churn_cleaned.csv')
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 7,043 rows x 29 columns


## 1. KPI Definitions & Computation

### KPI 1: Overall Churn Rate
**Formula:** (Customers Churned / Total Customers) x 100  
**Purpose:** Primary health metric — the headline number for executive reporting.

In [2]:
churn_rate = df['Churn_Binary'].mean() * 100
print(f"Overall Churn Rate: {churn_rate:.1f}%")
print(f"  Churned: {df['Churn_Binary'].sum():,}")
print(f"  Retained: {(df['Churn_Binary'] == 0).sum():,}")
print(f"  Total: {len(df):,}")

Overall Churn Rate: 26.5%
  Churned: 1,869
  Retained: 5,174
  Total: 7,043


### KPI 2: Monthly Revenue at Risk
**Formula:** Sum of MonthlyCharges for all churned customers  
**Purpose:** Quantifies the direct financial impact of churn.

In [3]:
monthly_rev_at_risk = df[df['Churn'] == 'Yes']['MonthlyCharges'].sum()
total_monthly_rev = df['MonthlyCharges'].sum()
rev_at_risk_pct = monthly_rev_at_risk / total_monthly_rev * 100

print(f"Monthly Revenue at Risk: ${monthly_rev_at_risk:,.2f}")
print(f"Total Monthly Revenue:   ${total_monthly_rev:,.2f}")
print(f"Revenue at Risk %:       {rev_at_risk_pct:.1f}%")
print(f"Annualized Revenue Loss: ${monthly_rev_at_risk * 12:,.2f}")

Monthly Revenue at Risk: $139,130.85
Total Monthly Revenue:   $456,116.60
Revenue at Risk %:       30.5%
Annualized Revenue Loss: $1,669,570.20


### KPI 3: Customer Lifetime Value (CLV) by Segment
**Formula:** Average TotalCharges per segment  
**Purpose:** Helps prioritize which customer segments to invest in retaining.

In [4]:
clv_by_segment = df.groupby('TenureBucket').agg(
    Avg_CLV=('TotalCharges', 'mean'),
    Median_CLV=('TotalCharges', 'median'),
    Count=('customerID', 'count'),
    Churn_Rate=('Churn_Binary', 'mean')
).round(2)

bucket_order = ['New (0-12)', 'Growing (13-24)', 'Mature (25-48)', 'Loyal (49-72)']
clv_by_segment = clv_by_segment.reindex(bucket_order)
clv_by_segment['Churn_Rate'] = (clv_by_segment['Churn_Rate'] * 100).round(1)

print("Customer Lifetime Value by Tenure Segment:")
print(clv_by_segment.to_string())

Customer Lifetime Value by Tenure Segment:
                 Avg_CLV  Median_CLV  Count  Churn_Rate
TenureBucket                                           
New (0-12)        275.23      170.88   2186        47.0
Growing (13-24)  1126.26     1148.55   1024        29.0
Mature (25-48)   2390.45     2402.57   1594        20.0
Loyal (49-72)    4685.51     4991.50   2239        10.0


### KPI 4: Average Revenue Per User (ARPU)
**Formula:** Average MonthlyCharges across all customers  
**Purpose:** Pricing benchmark and revenue density metric.

In [5]:
arpu_overall = df['MonthlyCharges'].mean()
arpu_retained = df[df['Churn'] == 'No']['MonthlyCharges'].mean()
arpu_churned = df[df['Churn'] == 'Yes']['MonthlyCharges'].mean()

print(f"ARPU (Overall):  ${arpu_overall:.2f}/month")
print(f"ARPU (Retained): ${arpu_retained:.2f}/month")
print(f"ARPU (Churned):  ${arpu_churned:.2f}/month")
print(f"\nChurned customers pay ${arpu_churned - arpu_retained:.2f} MORE per month on average.")

ARPU (Overall):  $64.76/month
ARPU (Retained): $61.27/month
ARPU (Churned):  $74.44/month

Churned customers pay $13.18 MORE per month on average.


### KPI 5: Service Adoption Rate
**Formula:** Percentage of internet customers subscribed to each add-on  
**Purpose:** Identifies upsell opportunities and their retention impact.

In [6]:
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']
internet_users = df[df['InternetService'] != 'No']

print("Service Adoption Rates (Internet Users Only):")
print("=" * 65)
print(f"{'Service':<20} {'Adoption %':>12} {'Churn w/':>10} {'Churn w/o':>10}")
print("-" * 65)
svc_data = []
for svc in service_cols:
    adopted = (internet_users[svc] == 'Yes').mean() * 100
    churn_with = internet_users[internet_users[svc] == 'Yes']['Churn_Binary'].mean() * 100
    churn_without = internet_users[internet_users[svc] == 'No']['Churn_Binary'].mean() * 100
    print(f"  {svc:<20} {adopted:>10.1f}% {churn_with:>9.1f}% {churn_without:>9.1f}%")
    svc_data.append({'Service': svc, 'Adoption_Rate': adopted, 
                     'Churn_With': churn_with, 'Churn_Without': churn_without})
print(f"\nTotal internet users: {len(internet_users):,}")

Service Adoption Rates (Internet Users Only):
Service                Adoption %   Churn w/  Churn w/o
-----------------------------------------------------------------
  OnlineSecurity             36.6%      14.6%      41.8%
  OnlineBackup               44.0%      21.5%      39.9%
  DeviceProtection           43.9%      22.5%      39.1%
  TechSupport                37.0%      15.2%      41.6%
  StreamingTV                49.1%      30.1%      33.5%
  StreamingMovies            49.5%      29.9%      33.7%

Total internet users: 5,517


### KPI 6: Contract Stability Index
**Formula:** Percentage of customers on 1-year or 2-year contracts  
**Purpose:** Measures overall customer commitment level.

In [7]:
long_term = (df['Contract'] != 'Month-to-month').mean() * 100
print(f"Contract Stability Index: {long_term:.1f}%")
print(f"  Month-to-month: {(df['Contract'] == 'Month-to-month').sum():,} ({(df['Contract'] == 'Month-to-month').mean()*100:.1f}%)")
print(f"  One year:       {(df['Contract'] == 'One year').sum():,} ({(df['Contract'] == 'One year').mean()*100:.1f}%)")
print(f"  Two year:       {(df['Contract'] == 'Two year').sum():,} ({(df['Contract'] == 'Two year').mean()*100:.1f}%)")

Contract Stability Index: 45.0%
  Month-to-month: 3,875 (55.0%)
  One year:       1,473 (20.9%)
  Two year:       1,695 (24.1%)


### KPI 7: High-Risk Customer Count & Revenue
**Formula:** Count of customers in the high-risk K-Means cluster  
**Purpose:** Sizes the retention campaign target audience.

In [8]:
if 'Cluster' in df.columns:
    cluster_stats = df.groupby('Cluster').agg(
        Count=('customerID', 'count'),
        Churn_Rate=('Churn_Binary', 'mean'),
        Monthly_Rev=('MonthlyCharges', 'sum')
    )
    cluster_stats['Churn_Rate'] = (cluster_stats['Churn_Rate'] * 100).round(1)
    
    high_risk_cluster = cluster_stats['Churn_Rate'].idxmax()
    hr = cluster_stats.loc[high_risk_cluster]
    
    print(f"High-Risk Cluster (Cluster {high_risk_cluster}):")
    print(f"  Customer Count:     {int(hr['Count']):,}")
    print(f"  Churn Rate:         {hr['Churn_Rate']:.1f}%")
    print(f"  Monthly Revenue:    ${hr['Monthly_Rev']:,.2f}")
    print(f"  Annualized Revenue: ${hr['Monthly_Rev'] * 12:,.2f}")
else:
    print("Note: Run notebook 04 first to generate cluster labels.")

High-Risk Cluster (Cluster 3):
  Customer Count:     1,782
  Churn Rate:         43.5%
  Monthly Revenue:    $155,179.85
  Annualized Revenue: $1,862,158.20


## 2. KPI Summary Table

In [9]:
kpi_summary = pd.DataFrame([
    {'KPI': 'Overall Churn Rate', 'Value': f'{churn_rate:.1f}%', 'Category': 'Health'},
    {'KPI': 'Monthly Revenue at Risk', 'Value': f'${monthly_rev_at_risk:,.0f}', 'Category': 'Financial'},
    {'KPI': 'Annualized Revenue Loss', 'Value': f'${monthly_rev_at_risk * 12:,.0f}', 'Category': 'Financial'},
    {'KPI': 'ARPU (Overall)', 'Value': f'${arpu_overall:.2f}', 'Category': 'Revenue'},
    {'KPI': 'ARPU (Churned)', 'Value': f'${arpu_churned:.2f}', 'Category': 'Revenue'},
    {'KPI': 'ARPU (Retained)', 'Value': f'${arpu_retained:.2f}', 'Category': 'Revenue'},
    {'KPI': 'Contract Stability Index', 'Value': f'{long_term:.1f}%', 'Category': 'Retention'},
    {'KPI': 'Total Customers', 'Value': f'{len(df):,}', 'Category': 'Scale'},
    {'KPI': 'Churned Customers', 'Value': f'{df["Churn_Binary"].sum():,}', 'Category': 'Scale'},
])
print(kpi_summary.to_string(index=False))

                     KPI      Value  Category
      Overall Churn Rate      26.5%    Health
 Monthly Revenue at Risk   $139,131 Financial
 Annualized Revenue Loss $1,669,570 Financial
          ARPU (Overall)     $64.76   Revenue
          ARPU (Churned)     $74.44   Revenue
         ARPU (Retained)     $61.27   Revenue
Contract Stability Index      45.0% Retention
         Total Customers      7,043     Scale
       Churned Customers      1,869     Scale


## 3. Tableau Data Exports

Exporting 4 purpose-built CSV files for Tableau dashboard creation.

In [10]:
# Export 1: Main dataset
export_cols = [col for col in df.columns if col != 'Churn_Binary']
df[export_cols].to_csv('../data/processed/tableau_main.csv', index=False)
print(f"1. tableau_main.csv: {df[export_cols].shape[0]:,} rows x {df[export_cols].shape[1]} columns")

1. tableau_main.csv: 7,043 rows x 28 columns


In [11]:
# Export 2: KPI Summary
kpi_summary.to_csv('../data/processed/tableau_kpi_summary.csv', index=False)
print(f"2. tableau_kpi_summary.csv: {len(kpi_summary)} KPIs")

2. tableau_kpi_summary.csv: 9 KPIs


In [12]:
# Export 3: Segment churn rates
segments = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'InternetService',
            'Contract', 'PaymentMethod', 'PaperlessBilling', 'TenureBucket', 
            'HighValueFlag', 'HasInternet']

segment_rows = []
for seg in segments:
    for val in df[seg].unique():
        subset = df[df[seg] == val]
        segment_rows.append({
            'Segment': seg,
            'Value': val,
            'Customer_Count': len(subset),
            'Churn_Count': subset['Churn_Binary'].sum(),
            'Churn_Rate': (subset['Churn_Binary'].mean() * 100).round(1),
            'Avg_Monthly_Charges': subset['MonthlyCharges'].mean().round(2),
            'Total_Monthly_Revenue': subset['MonthlyCharges'].sum().round(2),
            'Revenue_At_Risk': subset[subset['Churn'] == 'Yes']['MonthlyCharges'].sum().round(2)
        })

segment_df = pd.DataFrame(segment_rows)
segment_df.to_csv('../data/processed/tableau_segment_churn.csv', index=False)
print(f"3. tableau_segment_churn.csv: {len(segment_df)} segment-value combinations")
print(f"   Segments covered: {', '.join(segments)}")

3. tableau_segment_churn.csv: 28 segment-value combinations
   Segments covered: gender, SeniorCitizen, Partner, Dependents, InternetService, Contract, PaymentMethod, PaperlessBilling, TenureBucket, HighValueFlag, HasInternet


In [13]:
# Export 4: Cluster profiles
if 'Cluster' in df.columns:
    cluster_export = df.groupby('Cluster').agg(
        Customer_Count=('customerID', 'count'),
        Avg_Tenure=('tenure', 'mean'),
        Avg_Monthly_Charges=('MonthlyCharges', 'mean'),
        Avg_Total_Charges=('TotalCharges', 'mean'),
        Avg_NumServices=('NumServices', 'mean'),
        Pct_Month_to_Month=('ContractRisk', 'mean'),
        Churn_Rate=('Churn_Binary', 'mean'),
        Total_Monthly_Revenue=('MonthlyCharges', 'sum'),
        Revenue_At_Risk=('MonthlyCharges', lambda x: df.loc[x.index][df.loc[x.index, 'Churn'] == 'Yes']['MonthlyCharges'].sum())
    ).round(2)
    
    cluster_export['Churn_Rate'] = (cluster_export['Churn_Rate'] * 100).round(1)
    cluster_export['Pct_Month_to_Month'] = (cluster_export['Pct_Month_to_Month'] * 100).round(1)
    
    risk_order = cluster_export['Churn_Rate'].rank(method='dense')
    risk_map = {1.0: 'Low', 2.0: 'Medium-Low', 3.0: 'Medium-High', 4.0: 'High'}
    cluster_export['Risk_Level'] = risk_order.map(risk_map)
    
    cluster_export.to_csv('../data/processed/tableau_cluster_profiles.csv')
    print(f"4. tableau_cluster_profiles.csv: {len(cluster_export)} clusters")
    print(cluster_export.to_string())
else:
    print("4. Skipped — run notebook 04 first for cluster labels.")

4. tableau_cluster_profiles.csv: 4 clusters
         Customer_Count  Avg_Tenure  Avg_Monthly_Charges  Avg_Total_Charges  Avg_NumServices  Pct_Month_to_Month  Churn_Rate  Total_Monthly_Revenue  Revenue_At_Risk   Risk_Level
Cluster                                                                                                                                                                          
0                  1899       56.59                86.40            4954.04             4.15                 0.0        10.0              164077.80         17084.30   Medium-Low
1                  2094        9.20                48.82             429.82             0.58               100.0        42.0              102227.45         50337.95  Medium-High
2                  1268       40.01                27.31            1064.56             0.37                 0.0         3.0               34631.50          1218.85          Low
3                  1782       28.35                87.08          

## 4. Tableau Dashboard Field Mapping

Use this reference when building the Tableau dashboard:

### Dashboard 1: Executive Summary
| Visual | Data Source | Fields |
|---|---|---|
| KPI Cards | `tableau_kpi_summary.csv` | KPI, Value |
| Churn Donut | `tableau_main.csv` | COUNT(Churn) |
| Churn by Tenure | `tableau_segment_churn.csv` | Segment='TenureBucket', Value, Churn_Rate |
| Top Drivers | `tableau_segment_churn.csv` | Sorted by Churn_Rate descending |

### Dashboard 2: Customer Risk Segments
| Visual | Data Source | Fields |
|---|---|---|
| Cluster Scatter | `tableau_main.csv` | tenure, MonthlyCharges, Cluster (color) |
| Cluster Profiles | `tableau_cluster_profiles.csv` | All fields |
| Customer Table | `tableau_main.csv` | Filtered by Cluster selection |

### Dashboard 3: Service & Contract Deep-Dive
| Visual | Data Source | Fields |
|---|---|---|
| Service Heatmap | `tableau_segment_churn.csv` | Filter Segment IN (service columns) |
| Contract Bars | `tableau_segment_churn.csv` | Filter Segment='Contract' |
| Payment Method | `tableau_segment_churn.csv` | Filter Segment='PaymentMethod' |

### Dashboard 4: Revenue Impact
| Visual | Data Source | Fields |
|---|---|---|
| Revenue Treemap | `tableau_segment_churn.csv` | Revenue_At_Risk by Segment |
| High-Value Scatter | `tableau_main.csv` | MonthlyCharges, TotalCharges, Churn (color) |
| Retention Actions | Manual text/annotations | Based on analysis findings |

## 5. Final Verification

In [14]:
import os

export_files = [
    'tableau_main.csv',
    'tableau_kpi_summary.csv', 
    'tableau_segment_churn.csv',
    'tableau_cluster_profiles.csv'
]

print("Export Verification:")
print("=" * 60)
for f in export_files:
    path = f'../data/processed/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        rows = len(pd.read_csv(path))
        print(f"  {f:<35} {rows:>6} rows  {size/1024:>8.1f} KB")
    else:
        print(f"  {f:<35} MISSING")

print("\nAll exports complete. Ready for Tableau dashboard creation.")

Export Verification:
  tableau_main.csv                      7043 rows    1240.2 KB
  tableau_kpi_summary.csv                  9 rows       0.3 KB
  tableau_segment_churn.csv               28 rows       1.7 KB
  tableau_cluster_profiles.csv             4 rows       0.4 KB

All exports complete. Ready for Tableau dashboard creation.


## Summary

### KPIs Computed
1. **Overall Churn Rate** — Primary health metric
2. **Monthly Revenue at Risk** — Financial impact quantification
3. **Customer Lifetime Value** — By tenure segment
4. **ARPU** — Revenue density metric
5. **Service Adoption Rate** — Upsell opportunities
6. **Contract Stability Index** — Customer commitment level
7. **High-Risk Customer Count** — Retention campaign sizing

### Tableau Exports
4 CSV files exported to `data/processed/` — ready for import into Tableau Public.

---
*Next: Build Tableau Dashboard (manual step — see field mapping above)*